# Conversion Optimization with Bandits: Stop Wasting Traffic

**Goal:** Compare traditional A/B testing vs Thompson Sampling for landing page optimization. See how bandits save conversions by adapting allocation while learning.

**What you'll learn:**
- Why A/B testing has high opportunity cost (wasted conversions)
- How Thompson Sampling for Beta-Bernoulli converges faster
- How to calculate "lift" from using bandits vs A/B tests

**Time:** 15 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
print("Ready to optimize conversions!")

## Problem Setup: Landing Page Testing

You're testing 4 landing page variants for a commodity trading report signup:
- **Variant A:** "Professional Market Analysis" (3% conversion)
- **Variant B:** "Join 10,000 Traders" (6% conversion)
- **Variant C:** "Data-Driven Insights" (4% conversion)
- **Variant D:** "Trade Smarter" (5% conversion)

You have 10,000 visitors. Traditional A/B test would split 25% each. Bandits will adapt.

In [ ]:
# True conversion rates (unknown to us)
true_conversion_rates = {
    'A': 0.03,  # Worst
    'B': 0.06,  # Best
    'C': 0.04,
    'D': 0.05
}

variants = list(true_conversion_rates.keys())
n_visitors = 10000

def show_variant(variant_id):
    """Simulate showing a variant to a visitor"""
    true_rate = true_conversion_rates[variant_id]
    return 1 if np.random.rand() < true_rate else 0

print("Landing page variants:")
for v, rate in true_conversion_rates.items():
    print(f"  Variant {v}: {rate:.1%} conversion (true)")

## Baseline: Traditional A/B Test

Fixed 25% allocation to each variant for all 10,000 visitors.

In [ ]:
# A/B test: fixed allocation
ab_conversions = {v: 0 for v in variants}
ab_visitors = {v: 0 for v in variants}

for visitor in range(n_visitors):
    # Round-robin allocation (25% each)
    variant = variants[visitor % len(variants)]
    conversion = show_variant(variant)
    ab_visitors[variant] += 1
    ab_conversions[variant] += conversion

ab_total_conversions = sum(ab_conversions.values())

print("A/B TEST RESULTS:")
print("=" * 50)
for v in variants:
    obs_rate = ab_conversions[v] / ab_visitors[v]
    print(f"Variant {v}: {ab_visitors[v]} visitors, "
          f"{ab_conversions[v]} conversions ({obs_rate:.1%})")
print("=" * 50)
print(f"TOTAL: {ab_total_conversions} conversions\n")

# Calculate wasted conversions
best_rate = max(true_conversion_rates.values())
optimal_conversions = n_visitors * best_rate
wasted = optimal_conversions - ab_total_conversions
print(f"Optimal (all to best): {optimal_conversions:.0f} conversions")
print(f"A/B test waste: {wasted:.0f} conversions ({wasted/optimal_conversions:.1%})")

## Thompson Sampling: Adaptive Allocation

Beta-Bernoulli bandit that learns and tilts toward winners.

In [ ]:
class ThompsonSamplingBandit:
    def __init__(self, variants):
        self.variants = variants
        # Beta(1,1) prior = uniform
        self.alpha = {v: 1 for v in variants}
        self.beta = {v: 1 for v in variants}
        self.conversions = {v: 0 for v in variants}
        self.visitors = {v: 0 for v in variants}
        
    def select_variant(self):
        # Sample from each Beta posterior
        samples = {
            v: np.random.beta(self.alpha[v], self.beta[v]) 
            for v in self.variants
        }
        # Pick variant with highest sample
        return max(samples, key=samples.get)
    
    def update(self, variant, converted):
        self.visitors[variant] += 1
        if converted:
            self.alpha[variant] += 1
            self.conversions[variant] += 1
        else:
            self.beta[variant] += 1
    
    def get_posterior_mean(self, variant):
        a, b = self.alpha[variant], self.beta[variant]
        return a / (a + b)

# Run Thompson Sampling
bandit = ThompsonSamplingBandit(variants)
ts_history = []

for visitor in range(n_visitors):
    variant = bandit.select_variant()
    converted = show_variant(variant)
    bandit.update(variant, converted)
    
    # Track cumulative conversions
    ts_history.append(sum(bandit.conversions.values()))

ts_total_conversions = sum(bandit.conversions.values())

print("THOMPSON SAMPLING RESULTS:")
print("=" * 50)
for v in variants:
    visitors = bandit.visitors[v]
    convs = bandit.conversions[v]
    rate = convs / visitors if visitors > 0 else 0
    posterior = bandit.get_posterior_mean(v)
    print(f"Variant {v}: {visitors} visitors, "
          f"{convs} conversions ({rate:.1%}, "
          f"posterior: {posterior:.1%})")
print("=" * 50)
print(f"TOTAL: {ts_total_conversions} conversions\n")

# Calculate improvement
lift = (ts_total_conversions - ab_total_conversions) / ab_total_conversions
print(f"Thompson Sampling lift: +{lift:.1%} "
      f"({ts_total_conversions - ab_total_conversions:.0f} extra conversions)")

## Visualize: How Allocation Adapts Over Time

Watch Thompson Sampling shift traffic toward the best variant.

In [ ]:
# Re-run TS to track allocation over time
bandit2 = ThompsonSamplingBandit(variants)
allocation_history = {v: [] for v in variants}
window = 500  # Track allocation every 500 visitors

for visitor in range(n_visitors):
    variant = bandit2.select_variant()
    converted = show_variant(variant)
    bandit2.update(variant, converted)
    
    # Record allocation every window
    if (visitor + 1) % window == 0:
        total_v = sum(bandit2.visitors.values())
        for v in variants:
            pct = bandit2.visitors[v] / total_v
            allocation_history[v].append(pct)

# Plot allocation evolution
fig, ax = plt.subplots(figsize=(12, 6))
checkpoints = np.arange(window, n_visitors+1, window)

for v in variants:
    ax.plot(checkpoints, allocation_history[v], 
           marker='o', label=f"Variant {v} "
           f"(true: {true_conversion_rates[v]:.1%})", 
           linewidth=2)

ax.axhline(y=0.25, color='red', linestyle='--', 
          label='A/B test (fixed 25%)', alpha=0.5)
ax.set_xlabel('Visitors')
ax.set_ylabel('Traffic Allocation')
ax.set_title('Thompson Sampling: Adaptive Traffic Allocation')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0, 1])
plt.tight_layout()
plt.show()

print(f"\nFinal allocation:")
for v in variants:
    pct = bandit2.visitors[v] / n_visitors
    print(f"  Variant {v}: {pct:.1%}")

## Cumulative Conversions: A/B vs Bandit

The gap between the curves is money left on the table.

In [ ]:
# Regenerate cumulative for A/B test
ab_cumulative = []
ab_count = 0
for visitor in range(n_visitors):
    variant = variants[visitor % len(variants)]
    ab_count += show_variant(variant)
    ab_cumulative.append(ab_count)

# Regenerate for Thompson Sampling
bandit3 = ThompsonSamplingBandit(variants)
ts_cumulative = []
for visitor in range(n_visitors):
    variant = bandit3.select_variant()
    converted = show_variant(variant)
    bandit3.update(variant, converted)
    ts_cumulative.append(sum(bandit3.conversions.values()))

# Plot cumulative conversions
plt.figure(figsize=(12, 6))
plt.plot(ts_cumulative, label='Thompson Sampling', 
        linewidth=2, color='green')
plt.plot(ab_cumulative, label='A/B Test (fixed)', 
        linewidth=2, color='gray', alpha=0.7)

# Optimal baseline
optimal_line = np.arange(n_visitors) * best_rate
plt.plot(optimal_line, label='Optimal (all to best)', 
        linestyle='--', color='blue', alpha=0.5)

plt.xlabel('Visitors')
plt.ylabel('Cumulative Conversions')
plt.title('Conversions Over Time: Bandits vs A/B Testing')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

final_ab = ab_cumulative[-1]
final_ts = ts_cumulative[-1]
saved = final_ts - final_ab

print(f"\nFinal conversions:")
print(f"  A/B test: {final_ab}")
print(f"  Thompson Sampling: {final_ts}")
print(f"  Saved: {saved} conversions ({saved/final_ab:.1%} lift)")

## Posterior Distributions: What Did We Learn?

Visualize the Beta posteriors for each variant.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

x = np.linspace(0, 0.15, 1000)

for i, v in enumerate(variants):
    a = bandit3.alpha[v]
    b = bandit3.beta[v]
    true_rate = true_conversion_rates[v]
    
    # Plot Beta posterior
    pdf = stats.beta.pdf(x, a, b)
    axes[i].plot(x, pdf, linewidth=2, color='blue')
    axes[i].axvline(true_rate, color='red', 
                   linestyle='--', label=f'True: {true_rate:.1%}')
    axes[i].axvline(a/(a+b), color='green', 
                   linestyle='--', label=f'Estimate: {a/(a+b):.1%}')
    
    axes[i].set_title(f"Variant {v} Posterior\n"
                     f"Beta({a-1:.0f}, {b-1:.0f}) from "
                     f"{bandit3.visitors[v]} visitors")
    axes[i].set_xlabel('Conversion Rate')
    axes[i].set_ylabel('Density')
    axes[i].legend()
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Posterior means vs true rates:")
for v in variants:
    est = bandit3.get_posterior_mean(v)
    true = true_conversion_rates[v]
    print(f"  Variant {v}: {est:.2%} (true: {true:.2%})")

## Modify This: Experiment with Different Scenarios

Try these variations:

1. **Closer conversion rates:** Change to `[0.04, 0.045, 0.05, 0.055]` (harder problem)
2. **More variants:** Add 6 variants instead of 4
3. **Fewer visitors:** Change `n_visitors = 1000` (less data)
4. **Different priors:** Use `Beta(2, 20)` (assumes ~10% conversion)

**Questions to explore:**
- When are the gains biggest? (Easy problem with clear winner)
- When does Thompson Sampling struggle? (Many variants, little data)
- Does the prior help or hurt?

In [ ]:
# Your experiments here
# Modify true_conversion_rates and re-run

## Real-World Application: Commodity Report Signup

Let's simulate a realistic commodity trading use case.

In [ ]:
# Scenario: Testing 3 signup page headlines
commodity_variants = {
    'Professional Analysis': 0.028,
    'Join 5000 Traders': 0.055,
    'Energy Market Edge': 0.042
}

# Monthly traffic: 2,000 visitors
monthly_visitors = 2000

# A/B test: 4 weeks at 667 visitors each
ab_signups = sum(667 * rate for rate in commodity_variants.values())

# Thompson Sampling (simulation)
bandit_commodity = ThompsonSamplingBandit(
    list(commodity_variants.keys())
)

for _ in range(monthly_visitors):
    variant = bandit_commodity.select_variant()
    rate = commodity_variants[variant]
    converted = 1 if np.random.rand() < rate else 0
    bandit_commodity.update(variant, converted)

ts_signups = sum(bandit_commodity.conversions.values())
extra = ts_signups - ab_signups

print("COMMODITY REPORT SIGNUP OPTIMIZATION")
print("=" * 50)
print(f"A/B test signups (4 weeks): {ab_signups:.0f}")
print(f"Thompson Sampling signups: {ts_signups:.0f}")
print(f"Extra signups: {extra:.0f}")
print(f"\nValue (at $50/subscriber): ${extra * 50:.0f}\n")

print("Traffic allocation:")
for variant in commodity_variants.keys():
    pct = bandit_commodity.visitors[variant] / monthly_visitors
    print(f"  {variant}: {pct:.1%}")

## Summary

**Key Takeaways:**
1. **Thompson Sampling beats A/B testing** by 10-25% in cumulative conversions
2. **The gap grows over time** — bandits learn faster and waste less traffic
3. **Best for:** Clear winners, binary outcomes, moderate sample sizes
4. **For commodity traders:** Use this for report signups, alert channel testing, pricing experiments

**When bandits help most:**
- Large performance gap between variants (easy to detect winner)
- High opportunity cost per conversion (expensive traffic)
- Can't afford to waste 4+ weeks on fixed A/B test

**When bandits struggle:**
- Tiny differences between variants (need huge sample sizes)
- Too many variants (K > 10, slow to converge)
- Very noisy outcomes (hard to learn)

**Next:** Notebook 03 explores other business applications (pricing, email timing, onboarding flows).